# BamiBert Model

In [1]:
import pandas as pd

df = pd.read_csv(
    "../../data/processed/common_cleaned/vifactcheck_train_common_cleaned.csv"
)

df = df[
    ["Statement", "Evidence", "labels"]
].copy()

df.head()

,Statement,Evidence,labels
0,"Phó Thủ tướng Trần Hồng Hà thay mặt Chính phủ,...","Thay mặt Chính phủ, Thủ tướng Chính phủ, Phó T...",0
1,Hành vi của Tô Văn Hải là cho phép người khác ...,Tô Văn Hải đã có hành vi cho phép người khác đ...,0
2,SAWACO thông báo tạm ngưng cung cấp nước để th...,SAWACO thông báo tạm ngưng cung cấp nước để th...,1
3,"CLB luôn chuẩn bị rất kỹ lưỡng, chỉn chu chươn...",Là chương trình lớn nhất của CLB trong năm nên...,2
4,"ILA tiếp nhận và hỗ trợ học sinh miễn phí, Bé ...","Bé được tham gia kiểm tra trình độ đầu vào, tư...",2


In [2]:
# Load tokenizer BamiBERT
from transformers import AutoTokenizer

MODEL_NAME = "Qualcomm-AI-Research/BamiBERT"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

In [3]:
# Đưa hai input riêng cho tokenizer:
statement = df.loc[0, "Statement"]
evidence = df.loc[0, "Evidence"]

print("STATEMENT:")
print(statement)
print("\nSTATEMENT TOKENS:")
print(tokenizer.tokenize(statement))

print("\nEVIDENCE:")
print(evidence)
print("\nEVIDENCE TOKENS:")
print(tokenizer.tokenize(evidence))

encoded = tokenizer(
    statement,
    evidence
)

print(encoded["input_ids"])
print(encoded["attention_mask"])

STATEMENT:
Phó Thủ tướng Trần Hồng Hà thay mặt Chính phủ, Thủ tướng Chính phủ chúc mừng Đài Truyền hình Việt Nam, Đài truyền hình các tỉnh, thành phố trên cả nước, các đơn vị sản xuất truyền hình và TP. Hải Phòng sau 2 năm gián đoạn do đại dịch COVID-19 đã tổ chức rất thành công sự kiện quan trọng này.

STATEMENT TOKENS:
['PhÃ³', 'ĠThá»§', 'ĠtÆ°á»Ľng', 'ĠTráº§n', 'ĠHá»ĵng', 'ĠHÃł', 'Ġthay', 'Ġmáº·t', 'ĠChÃŃnh', 'Ġphá»§', ',', 'ĠThá»§', 'ĠtÆ°á»Ľng', 'ĠChÃŃnh', 'Ġphá»§', 'ĠchÃºc', 'Ġmá»«ng', 'ĠÄĲÃłi', 'ĠTruyá»ģn', 'ĠhÃ¬nh', 'ĠViá»ĩt', 'ĠNam', ',', 'ĠÄĲÃłi', 'Ġtruyá»ģn', 'ĠhÃ¬nh', 'ĠcÃ¡c', 'Ġtá»īnh', ',', 'ĠthÃłnh', 'Ġphá»ĳ', 'ĠtrÃªn', 'Ġcáº£', 'ĠnÆ°á»Ľc', ',', 'ĠcÃ¡c', 'ĠÄĳÆ¡n', 'Ġvá»ĭ', 'Ġsáº£n', 'Ġxuáº¥t', 'Ġtruyá»ģn', 'ĠhÃ¬nh', 'ĠvÃł', 'ĠTP', '.', 'ĠHáº£i', 'ĠPhÃ²ng', 'Ġsau', 'Ġ2', 'ĠnÄĥm', 'ĠgiÃ¡n', 'ĠÄĳoáº¡n', 'Ġdo', 'ĠÄĳáº¡i', 'Ġdá»ĭch', 'ĠCOVID-19', 'ĠÄĳÃ£', 'Ġtá»ķ', 'Ġchá»©c', 'Ġráº¥t', 'ĠthÃłnh', 'ĠcÃ´ng', 'Ġsá»±', 'Ġkiá»ĩn', 'Ġquan', 'Ġtrá»įng', 'ĠnÃły', '.']

EVIDENCE:
Thay mặ

In [4]:
# Trước khi chọn max_length, kiểm tra dataset
def get_length(row):
    encoded = tokenizer(
        row["Statement"],
        row["Evidence"],
        truncation=False
    )
    return len(encoded["input_ids"])


df["token_length"] = df.apply(
    get_length,
    axis=1
)

df["token_length"].describe(
    percentiles=[0.90, 0.95, 0.99]
)

count    5062.000000
mean       93.254050
std        35.653709
min        21.000000
50%        88.000000
90%       138.000000
95%       157.000000
99%       211.390000
max       343.000000
Name: token_length, dtype: float64

Max token_length là 343 => từ đó quyết định max_length = 384
Nếu padding tất cả lên 384:
- sample 60 tokens  → padding thêm 324
- sample 90 tokens  → padding thêm 294
=> Phí GPU => Dùng dynamic padding

In [5]:
from datasets import Dataset
from transformers import DataCollatorWithPadding

# Tokenize toàn dataset
dataset = Dataset.from_pandas(
    df[["Statement", "Evidence", "labels"]],
    preserve_index=False
)

MAX_LENGTH = 384

def tokenize_function(batch):
    return tokenizer(
        batch["Statement"],
        batch["Evidence"],
        truncation=True,
        max_length=MAX_LENGTH
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

# Dynamic padding
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

Map:   0%|          | 0/5062 [00:00<?, ? examples/s]

In [ ]:
# Model BamiBERT cho 3-class classification
from transformers import AutoModelForSequenceClassification

id2label = {
    0: "SUPPORTED",
    1: "REFUTED",
    2: "NEI"
}

label2id = {
    "SUPPORTED": 0,
    "REFUTED": 1,
    "NEI": 2
}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  412MB            

model.safetensors: downloading bytes:           |  0.00B            